In [1]:
# Cell 1 — Import Required Libraries and Configure LLM Access
import ollama

response = ollama.chat(
    model="llama3.1",
    messages=[
        {
            "role": "user",
            "content": "Why is the sky blue?"
        }
    ]
)

print(response["message"]["content"])

The sky appears blue because of a phenomenon called scattering, which is the way that light interacts with the tiny molecules of gases in the atmosphere.

Here's a simplified explanation:

1. **Sunlight enters Earth's atmosphere**: When sunlight enters our atmosphere, it consists of a spectrum of colors, including all the colors of the rainbow.
2. **Scattering occurs**: As sunlight travels through the atmosphere, it encounters tiny molecules of gases like nitrogen (N2) and oxygen (O2). These molecules are much smaller than the wavelength of light, so they scatter the light in all directions.
3. **Short wavelengths are scattered more**: The scattering effect is more pronounced for shorter wavelengths of light, such as blue and violet. This is known as Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described the phenomenon in the late 19th century.
4. **Blue light is scattered in all directions**: As a result of scattering, the blue light is dispersed in 

In [2]:
# Cell 2 — Load and Validate Cleaned Data
import pandas as pd
df = pd.read_csv("../data/processed/customer_support_tickets_clean.csv")

print(f"Duplicated tickets: {df.duplicated().sum()}")

invalid_type = (
    df["Ticket Type"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Invalid or empty Ticket Type values: {invalid_type}")

invalid_description = (
    df["Ticket Description"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Invalid or empty Ticket Description values: {invalid_description}")

Duplicated tickets: 0
Invalid or empty Ticket Type values: 0
Invalid or empty Ticket Description values: 0


In [3]:
# Cell 3 — Define Allowed Ticket Categories
ticket_categories = [
    "Technical Issue",
    "Refund Request",
    "Cancellation Request",
    "Billing Inquiry",
    "Product Inquiry"
]

In [4]:
# Cell 4 — Create the Structured Ticket Analysis Prompt

def create_prompt(ticket_description):
    prompt = f"""
Analyze the following customer-support ticket.

Ticket description:
{ticket_description}

Classify the ticket using exactly one of these categories:
- Technical Issue
- Refund Request
- Cancellation Request
- Billing Inquiry
- Product Inquiry

Return the analysis as JSON using exactly these four fields:

{{
    "Summary": "A short description of the customer's problem",
    "Category": "One of the five allowed ticket categories",
    "Sentiment": "The customer's expressed sentiment",
    "Urgency": "The estimated urgency of the problem"
}}

Return only the JSON object and no additional text.
"""
    return prompt

In [5]:
# Cell 5 — Create analyze_ticket() Function

import json

def analyze_ticket(ticket_description):
    if ticket_description.strip() == "":
        return None

    prompt = create_prompt(ticket_description)

    response = ollama.chat(
        model="llama3.1",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    response_text = response["message"]["content"]

    analysis = json.loads(response_text)

    return analysis

In [6]:
# Cell 6 — Select a Controlled Sample of Tickets

ticket_sample = pd.concat([
    df[df["Ticket Type"] == ticket_type].sample(
        5,
        random_state=42
    )
    for ticket_type in ticket_categories
]).reset_index(drop=True)

display(ticket_sample[["Ticket Description", "Ticket Type"]])

print("\nTickets per category:")
print(ticket_sample["Ticket Type"].value_counts())

,Ticket Description,Ticket Type
0,I'm having an issue with the {product_purchase...,Technical Issue
1,I'm having an issue with the {product_purchase...,Technical Issue
2,I'm having an issue with the {product_purchase...,Technical Issue
3,I've forgotten my password for my {product_pur...,Technical Issue
4,I'm having an issue with the {product_purchase...,Technical Issue
5,I'm facing issues logging into my {product_pur...,Refund Request
6,I'm having an issue with the {product_purchase...,Refund Request
7,I'm having an issue with the {product_purchase...,Refund Request
8,I'm unable to access my {product_purchased} ac...,Refund Request
9,I'm having an issue with the {product_purchase...,Refund Request



Tickets per category:
Ticket Type
Technical Issue         5
Refund Request          5
Cancellation Request    5
Billing Inquiry         5
Product Inquiry         5
Name: count, dtype: int64


In [7]:
# Cell 7 — Run LLM Analysis on the Sample

ticket_analysis = [
    analyze_ticket(description)
    for description in ticket_sample["Ticket Description"]
]

print(f"Tickets analyzed: {len(ticket_analysis)}")

Tickets analyzed: 25


In [8]:
# Cell 8 — Store Actual and LLM Results in a DataFrame

comparison = pd.DataFrame({
    "Ticket Description": ticket_sample["Ticket Description"],
    "Actual Type": ticket_sample["Ticket Type"],
    "LLM Category": [
        result["Category"]
        for result in ticket_analysis
    ],
    "Summary": [
        result["Summary"]
        for result in ticket_analysis
    ],
    "Sentiment": [
        result["Sentiment"]
        for result in ticket_analysis
    ],
    "Urgency": [
        result["Urgency"]
        for result in ticket_analysis
    ]
})

display(comparison)

,Ticket Description,Actual Type,LLM Category,Summary,Sentiment,Urgency
0,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Customer is having trouble finding an option i...,Neutral,Medium
1,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Customer is experiencing an issue with a product,Neutral,Medium
2,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Product upgrade issue after software update,Neutral,Medium
3,I've forgotten my password for my {product_pur...,Technical Issue,Technical Issue,Customer forgot password and reset option is n...,Help-seeking,High
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Customer concerned about security of {product_...,Neutral/Cautious,Medium
5,I'm facing issues logging into my {product_pur...,Refund Request,Technical Issue,"Account is locked and unable to log in, with t...",Frustrated/Help-seeking,High
6,I'm having an issue with the {product_purchase...,Refund Request,Technical Issue,Issue with {product_purchased} requiring assis...,Neutral,Medium
7,I'm having an issue with the {product_purchase...,Refund Request,Technical Issue,The customer is experiencing issues with the p...,Neutral,High
8,I'm unable to access my {product_purchased} ac...,Refund Request,Technical Issue,Customer unable to access account due to 'Inva...,Frustrated,High
9,I'm having an issue with the {product_purchase...,Refund Request,Technical Issue,Customer is experiencing an issue with the pro...,Negative,High


In [9]:
# Cell 9 — Inspect Correct and Incorrect LLM Classifications

comparison["Correct"] = (
    comparison["Actual Type"]
    == comparison["LLM Category"]
)

correct_classification = comparison[
    comparison["Correct"]
]

incorrect_classification = comparison[
    ~comparison["Correct"]
]

print("Correctly Classified Tickets:")
display(correct_classification)

print("Incorrectly Classified Tickets:")
display(incorrect_classification)

Correctly Classified Tickets:


,Ticket Description,Actual Type,LLM Category,Summary,Sentiment,Urgency,Correct
0,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Customer is having trouble finding an option i...,Neutral,Medium,True
1,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Customer is experiencing an issue with a product,Neutral,Medium,True
2,I'm having an issue with the {product_purchase...,Technical Issue,Technical Issue,Product upgrade issue after software update,Neutral,Medium,True
3,I've forgotten my password for my {product_pur...,Technical Issue,Technical Issue,Customer forgot password and reset option is n...,Help-seeking,High,True


Incorrectly Classified Tickets:


,Ticket Description,Actual Type,LLM Category,Summary,Sentiment,Urgency,Correct
4,I'm having an issue with the {product_purchase...,Technical Issue,Product Inquiry,Customer concerned about security of {product_...,Neutral/Cautious,Medium,False
5,I'm facing issues logging into my {product_pur...,Refund Request,Technical Issue,"Account is locked and unable to log in, with t...",Frustrated/Help-seeking,High,False
6,I'm having an issue with the {product_purchase...,Refund Request,Technical Issue,Issue with {product_purchased} requiring assis...,Neutral,Medium,False
7,I'm having an issue with the {product_purchase...,Refund Request,Technical Issue,The customer is experiencing issues with the p...,Neutral,High,False
8,I'm unable to access my {product_purchased} ac...,Refund Request,Technical Issue,Customer unable to access account due to 'Inva...,Frustrated,High,False
9,I'm having an issue with the {product_purchase...,Refund Request,Technical Issue,Customer is experiencing an issue with the pro...,Negative,High,False
10,I'm having an issue with the {product_purchase...,Cancellation Request,Technical Issue,The customer is experiencing an issue with the...,Neutral,Medium,False
11,I'm having an issue with the {product_purchase...,Cancellation Request,Technical Issue,Unresolved issue with product,Frustrated,High,False
12,I'm having an issue with the {product_purchase...,Cancellation Request,Technical Issue,Product is not charging properly with original...,Neutral,Medium,False
13,I'm having an issue with the {product_purchase...,Cancellation Request,Technical Issue,Customer is unable to find an option to perfor...,Neutral,Medium,False


In [10]:
# Cell 10 — Summarize LLM Results by Actual Ticket Type

correct_count = len(correct_classification)
incorrect_count = len(incorrect_classification)
total_count = len(comparison)

print(f"Total tickets: {total_count}")
print(f"Correct classifications: {correct_count}")
print(f"Incorrect classifications: {incorrect_count}")

category_summary = (
    comparison
    .groupby("Actual Type")["Correct"]
    .agg(["count", "sum"])
)

category_summary = category_summary.rename(
    columns={
        "count": "Total Tickets",
        "sum": "Correct Classifications"
    }
)

category_summary["Incorrect Classifications"] = (
    category_summary["Total Tickets"]
    - category_summary["Correct Classifications"]
)

display(category_summary)

Total tickets: 25
Correct classifications: 4
Incorrect classifications: 21


,Total Tickets,Correct Classifications,Incorrect Classifications
Actual Type,,,
Billing Inquiry,5,0,5
Cancellation Request,5,0,5
Product Inquiry,5,0,5
Refund Request,5,0,5
Technical Issue,5,4,1


In [11]:
# Cell 11 — Inspect Sentiment and Urgency Results

sentiment_summary = (
    comparison["Sentiment"]
    .value_counts()
)

urgency_summary = (
    comparison["Urgency"]
    .value_counts()
)

print("Sentiment Results:")
display(sentiment_summary)

print("Urgency Results:")
display(urgency_summary)

Sentiment Results:


Sentiment
Neutral                    12
Negative                    3
Neutral/Frustrated          2
Frustrated                  2
Help-seeking                1
Frustrated/Help-seeking     1
Neutral/Cautious            1
Concerned                   1
Concerned/Frustrated        1
Neutral/Concerned           1
Name: count, dtype: int64

Urgency Results:


Urgency
Medium    13
High      12
Name: count, dtype: int64